In [31]:
# import required libraries
from kafka import KafkaConsumer, KafkaProducer
import avro.schema
import avro.io
import io
import hashlib, json

In [32]:
def serialize(schema, obj):
    bytes_writer = io.BytesIO()
    encoder = avro.io.BinaryEncoder(bytes_writer)
    writer = avro.io.DatumWriter(schema)
    writer.write(obj, encoder)
    return bytes_writer.getvalue()

In [33]:
def deserialize(schema, raw_bytes):
    bytes_reader = io.BytesIO(raw_bytes)
    decoder = avro.io.BinaryDecoder(bytes_reader)
    reader = avro.io.DatumReader(schema)
    return reader.read(decoder)

In [34]:
schema_file = 'transaction.avsc'
txschema = avro.schema.parse(open(schema_file).read())
schema_file = 'submit.avsc'
submitschema = avro.schema.parse(open(schema_file).read())
schema_file = 'result.avsc'
resultschema = avro.schema.parse(open(schema_file).read())

In [35]:
# Connect to kafka broker running in your local host (docker). Change this to your kafka broker if needed
kafka_broker = 'lab.aimet.tech:9092'

In [36]:
producer = KafkaProducer(bootstrap_servers=[kafka_broker])

In [37]:
txconsumer = KafkaConsumer(
    'transaction',
     bootstrap_servers=[kafka_broker],
     enable_auto_commit=True,
     value_deserializer=lambda x: deserialize(txschema, x))
resultconsumer = KafkaConsumer(
    'result',
     bootstrap_servers=[kafka_broker],
     enable_auto_commit=True,
     value_deserializer=lambda x: deserialize(resultschema, x))

In [38]:
def gen_signature(txid, payer, payee, amount, token):
    o = {'txid': txid, 'payer': payer, 'payee': payee, 'amount': amount, 'token': token}
    return hashlib.md5(json.dumps(o, sort_keys=True).encode('utf-8')).hexdigest()

In [39]:
myVID = 'V263945'
myToken = '5daaf04f28df8a6aa9a1c45edc89b58d'

In [41]:
for tx_msg in txconsumer:
    tx = tx_msg.value
    txid = tx.get('txid')
    payer = tx.get('payer')
    payee = tx.get('payee')
    amount = tx.get('amount')

    signature = gen_signature(txid, payer, payee, amount, myToken)

    submit_obj = {'vid': myVID, 'txid': txid, 'signature': signature}
    submit_bytes = serialize(submitschema, submit_obj)
    producer.send('submit', submit_bytes)
    producer.flush()
    print('submitted verification:', submit_obj)

    print('waiting for result for txid:', txid)
    for res_msg in resultconsumer:
        res = res_msg.value
        if res.get('vid') == myVID and res.get('txid') == txid:
            print('received result:', res)
            if res.get('code') == 200:
                print('verification SUCCESS for', txid)
            else:
                print('verification FAILED for', txid, 'code=', res.get('code'))
            break
    print('-----------------------------------')

submitted verification: {'vid': 'V263945', 'txid': 'TX00622', 'signature': '63837499c9215f4fb157b990a8fb1b7f'}
waiting for result for txid: TX00622
received result: {'timestamp': 1763634841, 'vid': 'V263945', 'txid': 'TX00622', 'checksum': '09e7881e62318c85beeb0f1d62a86c07', 'code': 200, 'message': 'Confirm'}
verification SUCCESS for TX00622
-----------------------------------
submitted verification: {'vid': 'V263945', 'txid': 'TX02206', 'signature': 'e8b10c2be70a3c4dd15a401d8b54a162'}
waiting for result for txid: TX02206
received result: {'timestamp': 1763634842, 'vid': 'V263945', 'txid': 'TX02206', 'checksum': '153dce883a2315fd84a2e0de85c56a1b', 'code': 200, 'message': 'Confirm'}
verification SUCCESS for TX02206
-----------------------------------
submitted verification: {'vid': 'V263945', 'txid': 'TX06957', 'signature': '054ada7122f04a8c651a90e96439d7ac'}
waiting for result for txid: TX06957
received result: {'timestamp': 1763634843, 'vid': 'V263945', 'txid': 'TX06957', 'checksum': 

KeyboardInterrupt: 